# GPT-4 Response Context Depth Analysis

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [2]:
df_context = pd.read_csv('../Data/014/fct_context_retention.csv', parse_dates=['response_date'])

pl_context = pl.read_csv('../Data/014/fct_context_retention.csv', try_parse_dates=True)

# Pregunta 1

### ¿Cuál es el puntaje promedio de retención de contexto para las respuestas de GPT-4 en abril de 2024? Esto nos ayudará a determinar una medida base de la complejidad de las respuestas de GPT-4.

```SQL

SELECT
    ROUND(AVG(context_retention_score)::NUMERIC,2) AS avg_retention_score
FROM fct_context_retention
WHERE ((EXTRACT(MONTH FROM response_date) = 4) AND
       (EXTRACT(YEAR FROM response_date) = 2024)) AND
    (model_name = 'GPT-4');
```

In [10]:
abril = df_context[
    (df_context['response_date'].dt.month == 4) &
    (df_context['response_date'].dt.year == 2024) &
    (df_context['model_name'] == 'GPT-4')
].reset_index()

res = abril['context_retention_score'].mean().round(2)

In [9]:
res = pl_context.filter(
    (pl.col('response_date').dt.month() == 4) &
    (pl.col('response_date').dt.year() == 2024) &
    (pl.col('model_name') == 'GPT-4')
).select(
    pl.col('context_retention_score').mean().round(2).alias('avg_retention_score')
)

# Pregunta 2

### ¿Cuál es el puntaje de retención de contexto más alto registrado por GPT-4 para el tipo de consulta 'legal' en abril de 2024? Esto resaltará el desempeño máximo en términos de procesamiento contextual.

```SQL
SELECT
    MAX(context_retention_score) AS max_score
FROM fct_context_retention
WHERE ((EXTRACT(MONTH FROM response_date) = 4) AND
       (EXTRACT(YEAR FROM response_date) = 2024)) AND
    (model_name = 'GPT-4') AND
    (inquiry_type = 'legal')
```

In [12]:
abril = df_context[
    (df_context['response_date'].dt.month == 4) &
    (df_context['response_date'].dt.year == 2024) &
    (df_context['model_name'] == 'GPT-4') &
    (df_context['inquiry_type'] == 'legal')
].reset_index()

res = abril.agg(
    max_score = ('context_retention_score', 'max')
)

In [16]:
res = pl_context.filter(
    (pl.col('response_date').dt.month() == 4) &
    (pl.col('response_date').dt.year() == 2024) &
    (pl.col('model_name') == 'GPT-4') &
    (pl.col('inquiry_type') == 'legal')
).select(
    pl.col('context_retention_score').max().alias('max_score')
)

# Pregunta 3

### ¿Cuál es el puntaje promedio de retención de contexto para cada tipo de consulta en las respuestas de GPT-4 en abril de 2024, redondeado a dos decimales? Este desglose informará directamente qué dominios de consulta podrían necesitar mejoras en la comprensión contextual de GPT-4.

```SQL
SELECT
    inquiry_type,
    ROUND(AVG(context_retention_score)::NUMERIC,2) AS avg_score
FROM fct_context_retention
WHERE ((EXTRACT(MONTH FROM response_date) = 4) AND
       (EXTRACT(YEAR FROM response_date) = 2024)) AND
    (model_name = 'GPT-4')
GROUP BY inquiry_type
```

In [21]:
abril = df_context[
    (df_context['response_date'].dt.month == 4) &
    (df_context['response_date'].dt.year == 2024) &
    (df_context['model_name'] == 'GPT-4')
].groupby('inquiry_type').agg(
    avg_score = ('context_retention_score', 'mean')
).round(2).reset_index()

In [22]:
res = pl_context.filter(
    (pl.col('response_date').dt.month() == 4) &
    (pl.col('response_date').dt.year() == 2024) &
    (pl.col('model_name') == 'GPT-4') 
).group_by('inquiry_type').agg(
    pl.col('context_retention_score').mean().round(2).alias('avg_score')
)

res

inquiry_type,avg_score
str,f64
"""finance""",76.83
"""engineering""",88.6
"""legal""",86.53
"""tech""",93.03
"""health""",81.15
